# 01 · Extracción — IVR Alkosto (datos actuales)

Reemplaza al CSV histórico de Interaxa (`vw_interaxa_detalle_ivr_...csv`) que usaba
`01_Alk_final.ipynb`. Consulta directamente `vw_emt_gc_detalle_ivr` y aplana
`atributos_custom` en columnas `customN`.

**Salida:** `data/00_raw/df_raw_<fecha_inicio>_<fecha_fin>.parquet` — insumo del
Notebook 2 (`02_reconstruccion_traza.ipynb`).

**Requisitos:**
- Archivo `.env` en la raíz del proyecto (mismo nivel que este notebook o un nivel
  arriba) con las credenciales de conexión. Ver plantilla `.env.example` generada
  junto a este notebook — **no** subir el `.env` real a ningún repositorio.
- `pip install python-dotenv psycopg2-binary pandas pyarrow` (pyarrow para exportar
  a parquet; si prefieres solo Excel, puedes omitirlo y usar `.xlsx` al final).

In [ ]:
import os
import warnings
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## Parámetros del análisis

Ajusta aquí el periodo y la división antes de correr el resto del notebook. Todo lo
que sigue depende de estos valores — no hay rutas ni fechas quemadas más abajo.

In [ ]:
# --- Parámetros editables ---
FECHA_INICIO = "2026-06-01"
FECHA_FIN = "2026-06-30"

DIVISIONES = ["Alkosto", "Home,Alkosto", "Alkosto,Home"]

ORGANIZACION = "emtelcosas"

# Carpeta raíz de datos del proyecto (ajusta si tu estructura es distinta)
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
RAW_DIR = DATA_DIR / "00_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = RAW_DIR / f"df_raw_{FECHA_INICIO}_{FECHA_FIN}.parquet"
print(f"Salida esperada: {OUTPUT_PATH}")

## Conexión (credenciales desde `.env`, nunca hardcodeadas)

In [ ]:
load_dotenv()  # busca un archivo .env en el directorio actual o en los padres

DB_USER = os.getenv("IVR_DB_USER")
DB_PASSWORD = os.getenv("IVR_DB_PASSWORD")
DB_HOST = os.getenv("IVR_DB_HOST")
DB_PORT = int(os.getenv("IVR_DB_PORT", "5432"))
DB_NAME = os.getenv("IVR_DB_NAME")

faltantes = [
    nombre
    for nombre, valor in {
        "IVR_DB_USER": DB_USER,
        "IVR_DB_PASSWORD": DB_PASSWORD,
        "IVR_DB_HOST": DB_HOST,
        "IVR_DB_NAME": DB_NAME,
    }.items()
    if not valor
]
if faltantes:
    raise RuntimeError(
        f"Faltan variables de entorno en tu .env: {faltantes}. "
        "Revisa .env.example para ver los nombres esperados."
    )

In [ ]:
con = psycopg2.connect(
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
)

## Consulta parametrizada

Usa placeholders `%s` (psycopg2 los sustituye de forma segura) en vez de f-strings,
para no repetir el patrón de fechas/organización quemadas del notebook original.

In [ ]:
sql_query = """
SELECT *
FROM public.vw_emt_gc_detalle_ivr
WHERE organizacion = %(organizacion)s
  AND division = ANY(%(divisiones)s)
  AND fecha_inicio >= %(fecha_inicio)s
  AND fecha_inicio <= %(fecha_fin)s
ORDER BY fecha_inicio;
"""

params = {
    "organizacion": ORGANIZACION,
    "divisiones": DIVISIONES,
    "fecha_inicio": FECHA_INICIO,
    "fecha_fin": FECHA_FIN,
}

df = pd.read_sql_query(sql_query, con, params=params)
con.close()
print(df.shape)
df.head(3)

## Chequeos rápidos de sanidad

Antes de aplanar `atributos_custom`, valida que la extracción trajo lo esperado:
que no venga vacía, que las fechas caigan dentro del rango pedido y que
`id_conversacion` no tenga duplicados exactos.

In [ ]:
assert len(df) > 0, "La consulta no trajo filas — revisa parámetros de fecha/división."

print("Filas:", len(df))
print("id_conversacion únicos:", df["id_conversacion"].nunique())
print("Duplicados exactos de id_conversacion:", df["id_conversacion"].duplicated().sum())
print("Rango fecha_inicio:", df["fecha_inicio"].min(), "→", df["fecha_inicio"].max())
print("Divisiones encontradas:", df["division"].unique())
print("\nNulos en traza_opciones:", df["traza_opciones"].isna().sum())

## Aplanado de `atributos_custom`

Igual que en `consulta_ivr_alkosto.ipynb`: cada fila trae un dict con claves
`customN`, cada una con un único paso `codigo;texto;tiempo`. Los separamos en
columnas propias para poder reconstruir la traza completa en el Notebook 2.

In [ ]:
df_flat = df.join(df["atributos_custom"].apply(pd.Series))

columnas_custom = [c for c in df_flat.columns if c.startswith("custom")]
print(f"Columnas custom encontradas ({len(columnas_custom)}):", sorted(columnas_custom))
df_flat[["id_conversacion", "traza_opciones"] + columnas_custom].head(3)

## Exportar

Se guarda en `data/00_raw/` con el rango de fechas en el nombre, para poder tener
varias extracciones (distintos periodos) sin pisarlas entre sí. `parquet`
preserva mejor los tipos (fechas, dict) que Excel; si necesitas revisarlo a ojo,
exporta también una copia en `.xlsx` de una muestra pequeña.

In [ ]:
df_flat.to_parquet(OUTPUT_PATH, index=False)
print(f"Guardado: {OUTPUT_PATH}  ({len(df_flat)} filas)")

# Muestra legible en Excel para revisión manual rápida (primeras 200 filas)
muestra_path = RAW_DIR / f"df_raw_muestra_{FECHA_INICIO}_{FECHA_FIN}.xlsx"
df_flat.head(200).to_excel(muestra_path, index=False)
print(f"Muestra Excel: {muestra_path}")

---
**Siguiente paso:** `02_reconstruccion_traza.ipynb` — toma este parquet, une
`traza_opciones` + todas las columnas `customN` por `id_conversacion`, ordena por
código y reconstruye la traza completa equivalente a
`opcionesnavegaciontrazaopciones` del pipeline original.